# Implied Volatility Surface

In finance, volatility is a measure of how much an asset's returns change over time. Higher volatility means an asset's price moves up and down quickly, while a lower volatility means the asset's price doesn't change as quickly or steeply. Volatility is usually measured on an annual basis, but can be extended to longer or shorter time frames.

When we refer to volatility, we are usually referring to one of two types: Historical Volatility and Implied Volatility.

**Historical Volatility (HV)**

Historical volatility refers to the past volatility that an asset experienced. Historical volatility can be calculated by finding the annualized standard deviation of historical log returns. Here's how to do that:

Given a sequence of closing prices

$$
P_0, P_1, P_2, \ldots, P_n,
$$

the continuously compounded (log) return for day $i$ is

$$
r_i=\ln\left(\frac{P_i}{P_{i-1}}\right),
\qquad i=1,\ldots,n.
$$

The mean log return is

$$
\bar{r}
=
\frac{1}{n}
\sum_{i=1}^{n} r_i.
$$

The sample standard deviation of the log returns is

$$
\sigma_{\text{daily}}
=
\sqrt{
\frac{1}{n-1}
\sum_{i=1}^{n}
\left(r_i-\bar{r}\right)^2
}.
$$

The annualized historical volatility is

$$
\sigma_{\text{HV}}
=
\sigma_{\text{daily}}
\sqrt{252},
$$

where 252 is the approximate number of trading days in a year.

HV tells us how an asset behaved in the past, but what if we want to know how it is behaving right now?

**Implied Volatility**

Implied volatility is the market's estimate of the future volatility of an underlying asset
Unfortunately, historical volatility and the methods used to find it are not very useful when trying to infer the implied volatility of an asset. Past volatility is not indicative of future volatility, so we'll need a different strategy.

A very revolutionary tool used to deduce the implied volatility is the Black-Scholes equation. The equation was published by Fischer Black and Myron Scholes in 1973 and is used to deduce the fair price of options under several assumptions.

<div style="text-align: center;">
  <strong>Black-Scholes PDE</strong>
</div>                                        
$$
\frac{\partial V}{\partial t}
+ \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ rS\frac{\partial V}{\partial S}
- rV = 0.
$$
where

- $V(S,t)$ : Value of the derivative as a function of asset price and time
- $S$ : Current price of the underlying asset
- $t$ : Current time
- $\sigma$ : Volatility of the underlying asset
- $r$ : Continuously compounded risk-free interest rate 
- $\frac{\partial V}{\partial t}$ : Sensitivity of the option value to time
- $\frac{\partial V}{\partial S}$ : First derivative of the option value with respect to the asset price (Delta)
- $\frac{\partial^2 V}{\partial S^2}$ : Second derivative of the option value with respect to the asset price (Gamma)

In practice, computing with the PDE version is relatively slow, so let's change it into a closed-form version.

**First**, we need to assume that the underlying asset follows geometric brownian motion. 
$$dS = \mu S\,dt + \sigma S\,dW_t$$
where 

- $
\mu S\,dt
$   : Is the drift term (average growth rate), and
 
- $
\sigma S\,dW_t
$   : Is the random shock term (unpredictable noise)

This allows us to treat the Black-Scholes model as a stochastic one

**Second**, we apply Itô's Lemma to the option price $V(S,t)$,
assuming $V = V(S,t)$

Itô's Lemma gives:
$$
dV = V_t dt + V_s dS +\frac{1}{2} V_{ss}(dS)^2
$$
where

$
V_{SS} = \frac{\partial^2 V}{\partial S^2}
$


Now we can substitute 

$dS = \mu S\,dt + \sigma S\,dW$

$(dW)^2 = dt$


So: 
$$
dV = (V_t +\mu S\,V_s + \frac{1}{2}\sigma^2 S^2 V_{ss})dt + \sigma S\,V_s dW
$$


**Third**, we build a hedged portfolio to eliminate risk

We have a portfolio:
$$\Pi = V - \Delta S$$

where $V$ is the option price, $\Delta$ is the number of shares held short and $S$ is the stock price.

Taking the differential:

$$
d\Pi = dV - \Delta dS
$$

Substitute $dV$ and $dS$:

$$
d\Pi =
\left(
V_t + \mu S V_S + \frac{1}{2}\sigma^2 S^2 V_{SS}
\right)dt
+ \sigma S V_S dW
- \Delta (\mu S dt + \sigma S dW)
$$

and choose

$$
\Delta = V_S
$$

so that the random term $dW$ cancels. 


This removes all risk, making the portfolio deterministic.

Now we have:

$$
d\Pi = (V_t + \frac{1}{2}\sigma^2S^2V_{ss})dt
$$

Notice how the drift component $\mu$ and random component $dW$ are gone?
This means all risk is removed, making the portfolio deterministic.




**Fourth**, now that the portfolio $\Pi$ is risk free, it must grow at the risk free rate $r$.

So:
$$
d\Pi = r \Pi\, dt
$$

and 
$$
d\Pi = r(V - SV_s)dt
$$


We can equate both expressions

$$
V_t + \frac{1}{2}\sigma^2S^2V_{ss} = rV - rSV_s
$$

Rearrange

$$
V_t + \frac{1}{2}\sigma^2S^2V_{ss} + rSV_s - rV = 0
$$



**Fifth**, we anchor the PDE with $V(S,t) = max(S - K,0)$ at expiry (European call), and solve the PDE to obtain the closed form:
$$
C = N(d_+) S_t - N(d_-)Ke^{-rt}
$$

with

$
d_+ = \frac{ln(\frac{S_t}{K}) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}
$
,

$
d_- = d_+ - \sigma\sqrt{T}
$

and

$N$ being the cumulative distribution function (basically the area under the distribution up to d).


In [2]:
import numpy as np
import scipy.stats as sc
import plotly.io as pio
pio.renderers.default = "iframe_connected"
import plotly.graph_objects as go

from scipy.optimize import newton


def Black_Scholes_Call(S_t, K, r, T, sigma):

    d_p = (np.log(S_t/K) + (r + (sigma**2)/2)*(T))/(sigma*np.sqrt(T))

    d_m = d_p - sigma*np.sqrt(T)

    C = sc.norm.cdf(d_p)*S_t - sc.norm.cdf(d_m)*K*np.exp((-r)*(T))

    return C


def implied_volatility(C, S, K, r, T, sigma_init=0.20):

    def objective(sigma):
        return Black_Scholes_Call(S, K, r, T, sigma) - C
    implied_vol = newton(objective, sigma_init)

    return implied_vol


sigma_true = 0.25

data = []

for T in range(1, 6):
    for K in range(80, 121, 5):
        price = Black_Scholes_Call(100, K, 0.04, T, sigma_true)
        data.append((100, K, float(T), 0.04, price))


def implied_vol_surface(d):

    strikes = np.array([t[1] for t in d])
    maturities = np.array([t[2] for t in d])
    iv = np.array([implied_volatility(t[4], t[0], t[1], 0.034, t[2]) for t in d])

    uniq_strikes = np.unique(strikes)
    uniq_maturities = np.unique(maturities)

    S_grid = np.zeros((len(uniq_maturities), len(uniq_strikes)))
    T_grid = np.zeros((len(uniq_maturities), len(uniq_strikes)))
    IV_grid = np.zeros((len(uniq_maturities), len(uniq_strikes)))

    for i, T in enumerate(uniq_maturities):
        for j, K in enumerate(uniq_strikes):
            mask = (maturities == T) & (strikes == K)
            S_grid[i, j] = K
            T_grid[i, j] = T
            IV_grid[i, j] = iv[mask][0]
                
                      
    fig = go.Figure(data=[go.Surface(x=S_grid, y=T_grid, z=IV_grid)])

    fig.update_layout(
        title='Implied Volatility Surface',
        scene=dict(
            xaxis_title='Strike',
            yaxis_title='Maturity',
            zaxis_title='Implied Volatility'
        )
    )

    fig.show()

implied_vol_surface(data)